# Saving computation with relevance classifiers in long context filtering
This notebook demonstrates a basic procedure as follows:
1) One LLM labels a series of text chunks as relevant or potentially
irrelevant regarding the answering of a specific query.
2) Those labels are then used to train a classifier model, which is
actually another LLM which is split such that mid-network hidden layer
outputs are used to make binary classifications with a shallow
classification head.
3) This classifier is then effectively a less computationally intense
proxy for the former LLM and can then be used to effectively filter irrelevant
chunks almost as effectively as the original model.

By filtering irrelevant text, a downstream, more powerful LLM can answer
the original query with a less bloated context window as well as saving
FLOPs from the filtered tokens.

See `Saving_computation_with_relevance_classifiers_in_long_context_filtering.pdf`
for the full details of this procedure.

In this demonstration, we propose a case whre a Phi2 Model is doing the
first pass filtering to save computation from a Phi3 model. We estimate
that we save approximately 13.30% FLOPs over baseline by filtering 29.95% of
irrelevant chunks without incorrectly removing relevant chunks.

We use tasks from the Babilong dataset as examples of long text contexts
with questions answered by multihop reasoning through the text while
much of the text is irrelevant.

# Table of Contents
1. [Setup](#Setup)
2. [Execution](#Execution)
3. [Results](#Results)


## Setup
- Import modules
- Set constants
- Declare functions and classes

In [1]:
# Run on Google Colab with A100 runtime
!pip install transformers accelerate --quiet

import torch
import torch.nn as nn
import json
import requests
import pandas as pd
import os
import time
import numpy as np

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix
    )
from tqdm import tqdm
from typing import List, Dict, Tuple, Callable

# Mapping of string responses to boolean values
BOOL_MAP = {"yes": True, "no": False}

# Labelling functions

def set_seed(seed: int = 42) -> None:
    """
    Sets random seeds for reproducibility across NumPy and PyTorch
    operations.
    """
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def generate_boolean_response(
    labeler: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    prompt: str,
    max_new_tokens: int = 5,
    ) -> bool:
    """
    Used for labelling data.
    From a causal language model, given a prompt that asks to yes/no
    classify a given text chunk as relevant (yes/True) or irrelevant
    (no/False) to a query, outputs the True/False output.

    Args:
        labeler (AutoModelForCausalLM): Pretrained causal language model
        used for generating responses.
        tokenizer (AutoTokenizer): Pretrained tokenizer.
        prompt (str): Text prompt describing the context and question.
        max_new_tokens (int): Maximum number of tokens to generate in
        response. Default is 5.

    Returns:
        bool: True (is relevant), False (is not relevant)
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(labeler.device)
    input_len = inputs.input_ids.shape[-1]

    output = labeler.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    new_tokens = output[0][input_len:]
    generated_text = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
        ).strip()
    clean_output = generated_text.split()[0].lower()
    clean_output = clean_output if clean_output in ['yes', 'no'] else "yes"
    return BOOL_MAP[clean_output]

def get_relevance(
    labeler: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    now_task: Dict
    ) -> bool:
    """
    Determines whether a text chunk is relevant to a given question using
    an LLM.

    Args:
        labeler (AutoModelForCausalLM): Model used for relevance decision.
        now_task (Dict): Dictionary with 'chunk' and 'question' keys.

    Returns:
        bool: True if relevant, False otherwise.
    """
    prompt_text = f"""You are a relevance-detection assistant. Your job is to
decide whether the given context contains any information that could help
answer the question.

Respond only with:
- "yes" if there is any relevant information in the context
- "no" if the context contains no information that helps answer the question

Do not answer the question. Only say "yes" or "no".

Example 1:
Context: Sandra grabbed the apple. Then she went to the kitchen.
Question: Where was the apple before the kitchen?
Answer: yes

Example 2:
Context: Daniel went to the garden. Mary picked up the milk. John
walked to the bedroom.
Question: Where was the apple before the kitchen?
Answer: no

Task:
Context: {now_task['chunk']}
Question: {now_task['question']}
Answer:"""

    response = generate_boolean_response(labeler, tokenizer, prompt_text)
    return response

# Classifier functions and class

class NeuralNetClassifier(nn.Module):
    """
    Feedforward neural network for binary classification of embeddings.

    Args:
        input_dim (int): Size of input embeddings.
        hidden_dims (List[int]): Sizes of hidden layers.
        dropout_prob (float): Dropout probability between each hidden layer.

    Attributes:
        net (nn.Sequential): The sequence of layers used for feature
        extraction. classifier (nn.Linear): Output linear layer for
        binary classification.
    """

    def __init__(
        self,
        input_dim: int = 2560,
        hidden_dims: List[int] = [1024, 512, 256, 128],
        dropout_prob: float = 0.3
        ):
        super().__init__()
        layers = []
        for dim in hidden_dims:
            layers.append(nn.Linear(input_dim, dim))
            layers.append(nn.GELU())
            layers.append(nn.LayerNorm(dim))
            layers.append(nn.Dropout(dropout_prob))
            input_dim = dim
        self.net = nn.Sequential(*layers)
        self.classifier = nn.Linear(hidden_dims[-1], 1)

    def forward(
        self, x: torch.Tensor,
        ) -> Tuple[torch.Tensor, None, torch.Tensor]:
        """
        Performs a forward pass through the network.

        Args:
            x (torch.Tensor): Input embeddings tensor.
            sparsity_weight (float): Placeholder for sparsity loss weight.
                Default is 0.0.

        Returns:
            Tuple[torch.Tensor, None, torch.Tensor]:
                - Logits tensor (unactivated outputs).
                - Placeholder None (for compatibility).
                - Dummy sparsity loss (always zero).
        """
        z = self.net(x)
        logits = self.classifier(z).squeeze(-1)
        # x_hat and loss not used, but necessary for inheritance
        dummy_x_hat = None
        dummy_sparsity_loss = torch.tensor(0.0, device=x.device)
        return logits, dummy_x_hat, dummy_sparsity_loss


def encode_chunks(
    df: pd.DataFrame,
    tokenizer: AutoTokenizer,
    model: AutoModel,
    max_length: int = 256,
    layer_index: int = -2
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Encodes text chunks into embeddings using hidden states from a
    transformer model. Those embeddings are intended as inputs to an
    instance of NeuralNetClassifier which then makes binary
    classifications.

    Args:
        df (pd.DataFrame): DataFrame containing text chunks (in column
            labelled 'chunk') and labels (in column named 'is_relevant').
        tokenizer (AutoTokenizer): Tokenizer for the transformer model.
        model (AutoModel): Transformer model with hidden states enabled.
        max_length (int): Maximum sequence length for tokenization.
        layer_index (int): Which hidden layer to extract embeddings
            from model.

    Returns:
        Tuple[torch.Tensor, torch.Tensor]:
            - Embeddings tensor (num_samples x embedding_dim)
            - Label booleans as float32 tensor (num_samples)
    """
    embeddings, labels = [], []
    device = next(model.parameters()).device
    for _, row in tqdm(df.iterrows(), total=len(df)):
        tokens = tokenizer(
            row['chunk'],
            return_tensors='pt',
            truncation=True,
            padding="max_length",
            max_length=max_length
        ).to(device)
        with torch.no_grad():
            outputs = model(**tokens)
            hidden = outputs.hidden_states[layer_index]
            pooled = hidden.mean(dim=1).squeeze(0)
        embeddings.append(pooled.cpu())
        labels.append(row['is_relevant'])
    # boolean labels become float32 tensors for efficiency.
    return torch.stack(embeddings), torch.tensor(labels, dtype=torch.float32)

# Helper functions

def chunk_text(text: str, max_tokens: int = 500) -> List[str]:
    """
    Splits a long string into smaller chunks of approximately
    `max_tokens` words.

    Args:
        text (str): Input text to be split.
        max_tokens (int): Maximum number of words per chunk.

    Returns:
        List[str]: List of text chunks.
    """
    words = text.split()
    r = range(0, len(words), max_tokens)
    chunks = [" ".join(words[i:i + max_tokens]) for i in r]
    return chunks


def babilong_json_2_chunk_df(
    data_list: List[Dict],
    chunk_size: int
    ) -> pd.DataFrame:
    """
    Converts Babilong dataset JSON into a DataFrame of question-context
    chunks.

    Args:
        data_list (List[Dict]): List of task dictionaries containing
            'input', 'question', and 'target' keys.
        chunk_size (int): Maximum token count per chunk.

    Returns:
        pd.DataFrame: DataFrame with one row per chunk including metadata.
    """
    rows = []
    for qid, item in enumerate(data_list, start=1):
        input_text = item['input']
        question = item['question']
        target = item['target']
        chunks = chunk_text(input_text, max_tokens=chunk_size)
        for cid, chunk in enumerate(chunks, start=1):
            rows.append({
                'qid': qid,
                'cid': cid,
                'chunk': chunk,
                'question': question,
                'target': target
            })
    return pd.DataFrame(rows)


def get_labeled_chunks(
    labeler: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    tasks_df: pd.DataFrame,
    labeled_data_size: int,
    get_relevance: Callable,
    filename: str = "labeled_chunks.csv"
) -> pd.DataFrame:
    """
    Samples and labels a subset of task chunks for relevance classification.
    Uses LLM to produce binary labels and caches results to CSV.

    Args:
        labeler (AutoModelForCausalLM): Model used for labelling relevance.
        tasks_df (pd.DataFrame): DataFrame from `babilong_json_2_chunk_df`
        labeled_data_size (int): Number of samples to label.
        get_relevance (Callable): Function that performs LLM-based
            relevance detection.
        filename (str): CSV filename for caching labeled results.
            Operation skips and returns existing file if file already
            exists.

    Returns:
        pd.DataFrame: Labeled dataset with 'is_relevant' column.
    """
    if os.path.exists(filename):
        print("File already exists.")
        return pd.read_csv(filename)
    else:
        labeled_chunks = tasks_df.sample(
            n=labeled_data_size,
            random_state=42
            ).reset_index(drop=True)
        labeled_chunks['is_relevant'] = labeled_chunks.apply(
            lambda x: get_relevance(labeler, tokenizer, x),
            axis=1
            )
        labeled_chunks.to_csv(filename, index=False)
        return labeled_chunks


def find_optimal_threshold(
    y_true: np.ndarray,
    y_probs: np.ndarray,
    num_thresholds: int = 1000
    ) -> float:
    """
    Finds the optimal decision threshold that yields zero false negatives.

    Args:
        y_true (np.ndarray): Ground truth binary labels.
        y_probs (np.ndarray): Predicted probabilities.
        num_thresholds (int): Number of thresholds to test.

    Returns:
        float: Highest threshold with zero false negatives.
    """
    thresholds = np.linspace(0, 1, num_thresholds)
    best_threshold = 0.0
    for t in thresholds:
        y_pred = (y_probs >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        if fn > 0:
            break
        else:
            best_threshold = t
    return best_threshold


## Execution
- Set run-specific configurations
- Import and label data (skipped if file already exists)
- Train classifier
- Display performance on validation data

In [2]:
set_seed()

# Step 1: Configuration
data_url = "https://huggingface.co/datasets/RMT-team/babilong/resolve/main/data/qa3/4k.json"
chunk_size = 250                # Max number of tokens per chunk
labeled_data_size = 1000        # Number of samples to label
incision_layer = 28             # Layer where classifier will be inserted

# Step 2: Initialize device and model names
model_name = "microsoft/phi-2"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Step 3: Load tokenizer and language model for labeling
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
# Using the same kind of LLM for labeling and classification for convenience
labeler = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
    ).to(device)

# Step 4: Load base transformer for embedding extraction
C_base = AutoModel.from_pretrained(
    model_name,
    output_hidden_states=True
    ).to(device)
for param in C_base.parameters():
    param.requires_grad = False
C_base.eval()

# Step 5: Load dataset from Hugging Face repository
tasks_json = json.loads(requests.get(data_url).text)

# Step 6: Convert raw tasks to chunked DataFrame format
tasks_df = babilong_json_2_chunk_df(
    tasks_json[1:], # first entry is metadata, so we skip
    chunk_size=chunk_size
    )

# Step 7: Label random sample of chunks using LLM
labelling_time_start = time.time()
labeled_chunks = get_labeled_chunks(
    labeler,
    tokenizer,
    tasks_df,
    labeled_data_size,
    get_relevance
    ).dropna()
labelling_time_end = time.time()
print(f"Time taken for labelling: {labelling_time_end - labelling_time_start:.2f} seconds")

# Step 8: Split dataset into training, validation, and testing sets
train_val_df, test_df = train_test_split(
    labeled_chunks,
    test_size=0.2,
    random_state=42,
    stratify=labeled_chunks['is_relevant']
)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.1,
    random_state=42,
    stratify=train_val_df['is_relevant']
)

# Step 9: Encode chunks into embeddings using transformer model
train_embeddings, train_labels = encode_chunks(
    train_df,
    tokenizer,
    C_base,
    layer_index=incision_layer
    )
val_embeddings, val_labels = encode_chunks(
    val_df,
    tokenizer,
    C_base,
    layer_index=incision_layer
    )
test_embeddings, test_labels = encode_chunks(
    test_df,
    tokenizer,
    C_base,
    layer_index=incision_layer
    )

# Step 10: Move tensors to the appropriate device
train_embeddings, train_labels = train_embeddings.to(device), train_labels.to(device)
val_embeddings, val_labels = val_embeddings.to(device), val_labels.to(device)
test_embeddings, test_labels = test_embeddings.to(device), test_labels.to(device)

# Step 11: Initialize classifier model, optimizer, and loss function
model = NeuralNetClassifier(input_dim=train_embeddings.size(1)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.BCEWithLogitsLoss()

# Step 12: Training hyperparameters
batch_size = 64
epochs = 20
patience = 3


# init counter values
best_val_auc = 0
patience_counter = 0
start_train = time.time()

# Step 13: Train the classifier
for epoch in range(epochs):
    model.train()
    permutation = torch.randperm(train_embeddings.size(0))
    total_loss = 0

    for i in range(0, train_embeddings.size(0), batch_size):
        idx = permutation[i:i + batch_size]
        x_batch = train_embeddings[idx]
        y_batch = train_labels[idx]

        logits, _, sparsity_loss = model(x_batch)
        loss = loss_fn(logits.squeeze(), y_batch) + sparsity_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Step 14: Validation after each epoch
    model.eval()
    with torch.no_grad():
        val_logits, _, _ = model(val_embeddings)
        val_probs = torch.sigmoid(val_logits).squeeze()
        val_auc = roc_auc_score(val_labels.cpu(), val_probs.cpu())

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Val AUROC: {val_auc:.4f}")

    # Step 15: Early stopping based on validation AUROC
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

end_train = time.time()
print(f"Training time: {end_train - start_train:.2f} seconds")

# Step 16: Compute optimal classification threshold on validation set
optimal_threshold = find_optimal_threshold(
    val_labels.cpu().numpy(),
    val_probs.cpu().numpy()
    )
print(f"Optimal threshold (zero FP): {optimal_threshold:.3f}")

# Step 17: Evaluate model on test set
start_test = time.time()
model.eval()
with torch.no_grad():
    logits, _, _ = model(test_embeddings)
    probs = torch.sigmoid(logits).squeeze()
    preds = (probs > optimal_threshold).int() # extra certainty for test data

    test_labels_cpu = test_labels.cpu()
    preds_cpu = preds.cpu()
    probs_cpu = probs.cpu()

    acc = accuracy_score(test_labels_cpu, preds_cpu)
    precision = precision_score(test_labels_cpu, preds_cpu)
    recall = recall_score(test_labels_cpu, preds_cpu)
    auroc = roc_auc_score(test_labels_cpu, probs_cpu)

end_test = time.time()

# Step 18: Display evaluation metrics
print(f"Test Accuracy: {acc:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"AUROC: {auroc:.2f}")
print(f"Inference time: {end_test - start_test:.2f} seconds")




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Time taken for labelling: 164.97 seconds


100%|██████████| 200/200 [00:19<00:00, 10.27it/s]


Epoch 1, Loss: 3.3599, Val AUROC: 0.8827
Epoch 2, Loss: 2.8319, Val AUROC: 0.7627
Epoch 3, Loss: 2.7459, Val AUROC: 0.7947
Epoch 4, Loss: 2.6968, Val AUROC: 0.7973
Early stopping triggered.
Training time: 0.48 seconds
Optimal threshold (zero FP): 0.021
Test Accuracy: 0.34
Precision: 0.08
Recall: 0.92
AUROC: 0.78
Inference time: 0.01 seconds


## Results
Here we see that we correctly identified 56 out of 187 irrelevant chunks
without incorrectly classifying any relevant chunks as irrelevant.

The main manuscript steps through the calculations, which show that this
would yield a roughly 13.30% reduction in FLOPs versus using only
a Phi3 for query-answering.

In [3]:
# Step 19: Display confusion matrix
cm = confusion_matrix(test_labels_cpu, preds_cpu)
labels = ['Negative', 'Positive']
cm_df = pd.DataFrame(
    cm,
    index=[f'True {label}' for label in labels],
    columns=[f'Pred {label}' for label in labels]
    )

print("Confusion Matrix:")
print(cm_df)

Confusion Matrix:
               Pred Negative  Pred Positive
True Negative             56            131
True Positive              1             12
